## Example 1: stumpy functionality test 1

In [ ]:
import stumpy
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Example 1: Basic STUMPY matrix profile computation
print("=== Example 1: Basic Matrix Profile Computation ===")

# STUMPY is a powerful library for matrix profile computation and time series analysis
print(f"STUMPY version: {stumpy.__version__}")

# Create synthetic time series with patterns
np.random.seed(42)
n_points = 100
base_pattern = np.array([1, 2, 3, 2, 1, 0, -1, 0])
noise_level = 0.1

# Generate time series with repeated patterns and noise
ts = []
for i in range(n_points // len(base_pattern) + 1):
    pattern = base_pattern + np.random.normal(0, noise_level, len(base_pattern))
    ts.extend(pattern)

ts = np.array(ts[:n_points])
ts += np.random.normal(0, 0.05, len(ts))  # Add overall noise

print(f"Synthetic time series created:")
print(f"  Length: {len(ts)} points")
print(f"  Base pattern length: {len(base_pattern)}")
print(f"  Data range: [{ts.min():.3f}, {ts.max():.3f}]")

# Compute matrix profile using STUMP algorithm
m = len(base_pattern)  # Subsequence length (window size)
print(f"\nComputing matrix profile with window size m = {m}")

# STUMP is the core algorithm for matrix profile computation
try:
    matrix_profile = stumpy.stump(ts, m=m)
    
    print(f"Matrix profile computation successful!")
    print(f"Matrix profile shape: {matrix_profile.shape}")
    print(f"Matrix profile columns: distance, index")
    
    # Extract distances and nearest neighbor indices
    # stump returns ONE object-dtype array (distances and indices side by side), and numpy cannot take the square
    # root of an object array, so .std() on a column of it raises TypeError. P_ and I_ are stumpy's own typed views
    # of the two columns (float64 and int64).
    distances = matrix_profile.P_
    nn_indices = matrix_profile.I_
    
    print(f"\nMatrix profile analysis:")
    print(f"  Minimum distance: {distances.min():.4f}")
    print(f"  Maximum distance: {distances.max():.4f}")
    print(f"  Mean distance: {distances.mean():.4f}")
    print(f"  Distance standard deviation: {distances.std():.4f}")
    
    # Find the most similar subsequence pair (top motif)
    motif_idx = np.argmin(distances)
    motif_distance = distances[motif_idx]
    motif_nn_idx = int(nn_indices[motif_idx])
    
    print(f"\nTop motif (most similar pair):")
    print(f"  Position 1: {motif_idx}")
    print(f"  Position 2: {motif_nn_idx}")
    print(f"  Distance: {motif_distance:.4f}")
    
except Exception as e:
    print(f"Error in matrix profile computation: {e}")
    raise

print(f"\nSTUMP algorithm features:")
print(f"  - Exact matrix profile computation")
print(f"  - O(n²) time complexity")  
print(f"  - Finds all pairwise distances between subsequences")
print(f"  - Enables motif and discord discovery")

## Example 2: stumpy functionality test 2

In [ ]:
import stumpy
import numpy as np

# Example 2: Time series motif discovery with STUMPY
print("=== Example 2: Time series motif discovery ===")

# Create a more complex time series with multiple patterns
np.random.seed(123)

# Define different pattern types
sine_pattern = np.sin(np.linspace(0, 2*np.pi, 20))
triangle_pattern = np.concatenate([np.linspace(-1, 1, 10), np.linspace(1, -1, 10)])
step_pattern = np.array([-1]*5 + [0]*5 + [1]*5 + [0]*5)

# Create composite time series with multiple instances of patterns
ts_complex = []

# Add patterns in sequence with noise
for i in range(3):
    ts_complex.extend(sine_pattern + np.random.normal(0, 0.1, len(sine_pattern)))
    ts_complex.extend(triangle_pattern + np.random.normal(0, 0.1, len(triangle_pattern)))
    ts_complex.extend(step_pattern + np.random.normal(0, 0.1, len(step_pattern)))
    
    # Add some random noise between patterns
    ts_complex.extend(np.random.normal(0, 0.5, 10))

ts_complex = np.array(ts_complex)

print(f"Complex time series created:")
print(f"  Total length: {len(ts_complex)} points")
print(f"  Contains multiple pattern types repeated 3 times each")
print(f"  Pattern lengths: sine=20, triangle=20, step=20")

# Compute matrix profile for motif discovery
m = 20  # Use pattern length as window size
print(f"\nComputing matrix profile with window size m = {m}")

try:
    matrix_profile = stumpy.stump(ts_complex, m=m)
    
    # Use STUMPY's motif discovery function
    motifs = stumpy.motifs(ts_complex, matrix_profile[:, 0], max_motifs=5)
    
    print(f"\nTop motifs discovered:")
    print(f"Number of motifs found: {len(motifs)}")
    
    for i, motif_pair in enumerate(motifs):
        if len(motif_pair) >= 2:  # Ensure we have at least 2 positions
            pos1, pos2 = motif_pair[0], motif_pair[1] 
            # Calculate distance between the motif pair
            subseq1 = ts_complex[pos1:pos1+m]
            subseq2 = ts_complex[pos2:pos2+m]
            distance = np.linalg.norm(subseq1 - subseq2)
            
            print(f"  Motif {i+1}:")
            print(f"    Positions: {pos1}, {pos2}")
            print(f"    Distance: {distance:.4f}")
            print(f"    Pattern type: {'sine' if pos1 % 60 < 20 else 'triangle' if pos1 % 60 < 40 else 'step'}")

except Exception as e:
    print(f"Error in motif discovery: {e}")
    raise

# Demonstrate discord discovery (anomalies)
print(f"\nDiscord discovery (anomalies):")
try:
    # Find discords - subsequences that are most dissimilar to all others
    distances = matrix_profile[:, 0]
    
    # Top 3 discords (highest distances)
    discord_indices = np.argsort(distances)[-3:][::-1]
    
    print(f"Top discords (anomalous patterns):")
    for i, discord_idx in enumerate(discord_indices):
        discord_distance = distances[discord_idx]
        print(f"  Discord {i+1}:")
        print(f"    Position: {discord_idx}")
        print(f"    Distance: {discord_distance:.4f}")
        print(f"    Likely anomaly or transition region")

except Exception as e:
    print(f"Error in discord discovery: {e}")
    raise

# Demonstrate pattern matching
print(f"\nPattern matching capabilities:")
try:
    # Extract a query pattern (first sine wave)
    query_pattern = ts_complex[0:20]
    
    # Use mass to find all occurrences
    distances = stumpy.mass(query_pattern, ts_complex)
    
    # Find similar patterns (low distances)
    similar_positions = np.where(distances < 0.5)[0]
    
    print(f"Query pattern length: {len(query_pattern)}")
    print(f"Similar patterns found at positions: {similar_positions[:10]}")  # Show first 10
    print(f"Total similar patterns: {len(similar_positions)}")

except Exception as e:
    print(f"Error in pattern matching: {e}")
    raise

print(f"\nSTUMPY motif discovery enables:")
print(f"  - Finding repeated patterns automatically")
print(f"  - Detecting anomalies and outliers")
print(f"  - Pattern matching and similarity search")
print(f"  - Time series clustering and classification")

## Example 3: stumpy functionality test 3

In [ ]:
import stumpy
import numpy as np

# Example 3: Approximate algorithms and streaming analysis
print("=== Example 3: Approximate Algorithms and Streaming Analysis ===")

# Create a larger time series for performance testing
np.random.seed(456)
n_large = 1000
large_ts = np.sin(np.linspace(0, 10*np.pi, n_large)) + 0.5*np.sin(np.linspace(0, 50*np.pi, n_large))
large_ts += np.random.normal(0, 0.1, n_large)

print(f"Large time series created:")
print(f"  Length: {len(large_ts)} points")
print(f"  Multiple frequency components with noise")

# Demonstrate SCRUMP - approximate matrix profile
m = 50
print(f"\nTesting SCRUMP (approximate) vs STUMP (exact) algorithms:")
print(f"Window size: {m}")

# SCRUMP for approximate results
try:
    print(f"\nSCRUMP (approximate algorithm):")
    scrump_result = stumpy.scrump(large_ts, m=m, percentage=0.1)  # Use 10% of data
    
    # Get the current state
    scrump_mp = scrump_result.P_  # Matrix profile distances
    scrump_indices = scrump_result.I_  # Matrix profile indices
    
    print(f"  Algorithm: Approximate with 10% sample")
    print(f"  Matrix profile shape: {scrump_mp.shape}")
    print(f"  Computation faster than exact algorithm")
    print(f"  Min distance: {scrump_mp.min():.4f}")
    print(f"  Max distance: {scrump_mp.max():.4f}")
    print(f"  Mean distance: {scrump_mp.mean():.4f}")
    
except Exception as e:
    print(f"Error with SCRUMP: {e}")
    raise

# Demonstrate streaming analysis with STUMPI
print(f"\nStreaming analysis with STUMPI:")
try:
    # Initialize streaming matrix profile
    initial_ts = large_ts[:200]  # Start with first 200 points
    streaming_mp = stumpy.stumpi(initial_ts, m=m)
    
    print(f"  Initial time series length: {len(initial_ts)}")
    print(f"  Initial matrix profile length: {len(streaming_mp.P_)}")
    
    # Add new data points incrementally
    new_points = large_ts[200:220]  # Add 20 new points
    
    for new_point in new_points:
        streaming_mp.update(new_point)
    
    print(f"  Added {len(new_points)} new points incrementally")
    print(f"  Updated matrix profile length: {len(streaming_mp.P_)}")
    print(f"  Streaming enables real-time pattern analysis")
    
except Exception as e:
    print(f"Error with streaming analysis: {e}")
    raise

# Demonstrate multi-dimensional time series analysis
print(f"\nMulti-dimensional time series analysis (MSTUMP):")
try:
    # Create multi-dimensional time series (3 dimensions)
    n_dims = 3
    n_points_md = 200
    
    # Each dimension has different patterns
    md_ts = np.zeros((n_points_md, n_dims))
    md_ts[:, 0] = np.sin(np.linspace(0, 4*np.pi, n_points_md))  # Sine wave
    md_ts[:, 1] = np.cos(np.linspace(0, 3*np.pi, n_points_md))  # Cosine wave
    md_ts[:, 2] = np.linspace(-1, 1, n_points_md)               # Linear trend
    
    # Add noise to all dimensions
    md_ts += np.random.normal(0, 0.1, (n_points_md, n_dims))
    
    print(f"  Multi-dimensional time series shape: {md_ts.shape}")
    
    # Compute multi-dimensional matrix profile
    m_md = 20
    # mstump wants ONE ROW PER DIMENSION (shape dims x points), so it gets the transpose, and it returns two arrays:
    # P, one matrix profile per number of dimensions combined (row 0 = best single dimension), and I, their indices.
    md_P, md_I = stumpy.mstump(md_ts.T, m=m_md)
    
    print(f"  Window size: {m_md}")
    print(f"  Multi-dimensional matrix profile shape: {md_P.shape}")
    print(f"  Finds patterns across all dimensions simultaneously")
    print(f"  Min distance: {md_P[0].min():.4f}")
    print(f"  Mean distance: {md_P[0].mean():.4f}")
    
except Exception as e:
    print(f"Error with multi-dimensional analysis: {e}")
    raise

# Show algorithm comparison
print(f"\nSTUMPY algorithm comparison:")
algorithms = {
    'STUMP': 'Exact algorithm, O(n²) time, best accuracy',
    'SCRUMP': 'Approximate algorithm, faster, good accuracy',
    'STUMPI': 'Streaming algorithm, incremental updates',
    'MSTUMP': 'Multi-dimensional, handles multiple variables'
}

for algo, description in algorithms.items():
    availability = "✓" if hasattr(stumpy, algo.lower()) or hasattr(stumpy, algo) else "✗"
    print(f"  {algo}: {description} [{availability}]")

print(f"\nChoose algorithm based on:")
print(f"  - Data size (SCRUMP for large datasets)")
print(f"  - Real-time requirements (STUMPI for streaming)")
print(f"  - Accuracy needs (STUMP for exact results)")
print(f"  - Multiple variables (MSTUMP for multivariate)")
print(f"  - Computational resources (GPU variants for acceleration)")

## Example 4: stumpy functionality test 4

In [ ]:
import stumpy
import numpy as np

# Example 4: Advanced pattern analysis and time series segmentation  
print("=== Example 4: Advanced Pattern Analysis and Segmentation ===")

# Create complex time series with regime changes
np.random.seed(789)

# Define different regimes/phases
regime1 = np.sin(np.linspace(0, 6*np.pi, 150)) * 0.5  # Low amplitude sine
regime2 = np.sin(np.linspace(0, 12*np.pi, 150)) * 1.5  # High amplitude, higher frequency
regime3 = np.cumsum(np.random.normal(0, 0.1, 150))  # Random walk
regime4 = np.repeat([1, -1, 0.5, -0.5], 37)[:148]  # Step pattern

# Combine regimes with transitions
complex_ts = np.concatenate([regime1, regime2, regime3, regime4])

print(f"Complex time series with regime changes:")
print(f"  Total length: {len(complex_ts)} points")
print(f"  Regime 1 (0-149): Low amplitude sine wave")
print(f"  Regime 2 (150-299): High amplitude, high frequency sine")
print(f"  Regime 3 (300-449): Random walk")
print(f"  Regime 4 (450-597): Step pattern")

# Use FLUSS for semantic segmentation
print(f"\nFLUSS (Fast Low-Cost Unipotent Semantic Segmentation):")
try:
    m = 30  # Window size for segmentation
    
    # Compute FLUSS for regime change detection. FLUSS reads a matrix profile INDEX, not the series: stump gives it,
    # the window size is FLUSS's L, and this series has 4 regimes. It returns the corrected arc curve (low values
    # mark a regime change) and the change locations it picked.
    fluss_result, regime_locations = stumpy.fluss(stumpy.stump(complex_ts, m=m).I_, L=m, n_regimes=4)
    
    print(f"  Window size: {m}")
    print(f"  FLUSS profile length: {len(fluss_result)}")
    print(f"  FLUSS finds regime change points automatically")
    
    # Find potential regime change points (local minima in FLUSS profile)
    # Lower values in FLUSS indicate potential change points
    change_threshold = np.percentile(fluss_result, 10)  # Bottom 10%
    potential_changes = np.where(fluss_result < change_threshold)[0]
    
    # Filter change points that are too close together
    min_distance = 50  # Minimum distance between change points
    filtered_changes = []
    for change_point in potential_changes:
        if not filtered_changes or (change_point - filtered_changes[-1]) >= min_distance:
            filtered_changes.append(change_point)
    
    print(f"  Detected change points at positions: {filtered_changes}")
    print(f"  Expected change points around: [150, 300, 450]")
    
    # Evaluate change point accuracy
    expected_changes = [150, 300, 450]
    print(f"  Change point detection accuracy:")
    for expected in expected_changes:
        closest_detected = min(filtered_changes, key=lambda x: abs(x - expected)) if filtered_changes else None
        if closest_detected:
            error = abs(closest_detected - expected)
            print(f"    Expected {expected}, detected {closest_detected}, error: {error}")

except Exception as e:
    print(f"Error with FLUSS segmentation: {e}")
    raise

# Demonstrate AAMP for annotation vector analysis
print(f"\nAAMP (Annotation Vector Analysis):")
try:
    # Create annotation vector (different pattern types)
    annotations = []
    annotations.extend([0] * 150)  # Regime 1: sine wave
    annotations.extend([1] * 150)  # Regime 2: high freq sine
    annotations.extend([2] * 150)  # Regime 3: random walk
    annotations.extend([3] * len(regime4))  # Regime 4: step pattern
    
    annotation_vector = np.array(annotations)
    
    print(f"  Annotation vector length: {len(annotation_vector)}")
    print(f"  Unique annotations: {np.unique(annotation_vector)}")
    
    # Compute AAMP matrix profile
    m_aamp = 25
    aamp_result = stumpy.aamp(complex_ts, m=m_aamp)
    
    print(f"  AAMP matrix profile computed")
    print(f"  Window size: {m_aamp}")
    print(f"  AAMP enables annotation-aware pattern discovery")
    print(f"  Finds patterns within and across different regimes")

except Exception as e:
    print(f"Error with AAMP analysis: {e}")
    raise

# Demonstrate subsequence matching across regimes
print(f"\nCross-regime pattern matching:")
try:
    # Extract patterns from different regimes
    pattern_regime1 = complex_ts[20:50]    # Pattern from regime 1
    pattern_regime2 = complex_ts[170:200]  # Pattern from regime 2
    
    print(f"  Pattern 1 from regime 1: length {len(pattern_regime1)}")
    print(f"  Pattern 2 from regime 2: length {len(pattern_regime2)}")
    
    # Find similar patterns throughout the entire series
    distances1 = stumpy.mass(pattern_regime1, complex_ts)
    distances2 = stumpy.mass(pattern_regime2, complex_ts)
    
    # Find best matches for each pattern
    best_match1 = np.argmin(distances1)
    best_match2 = np.argmin(distances2)
    
    print(f"  Best match for regime 1 pattern at position: {best_match1}")
    print(f"  Best match for regime 2 pattern at position: {best_match2}")
    print(f"  Cross-regime matching distance 1: {distances1[best_match1]:.4f}")
    print(f"  Cross-regime matching distance 2: {distances2[best_match2]:.4f}")

except Exception as e:
    print(f"Error in cross-regime matching: {e}")
    raise

# Show advanced analysis capabilities
print(f"\nAdvanced STUMPY capabilities:")
advanced_features = [
    "Semantic segmentation (FLUSS): Automatic regime change detection",
    "Annotation-aware analysis (AAMP): Pattern discovery with labels", 
    "Streaming updates (STUMPI): Real-time pattern monitoring",
    "Multi-dimensional analysis: Patterns across multiple variables",
    "GPU acceleration: Fast computation for large datasets",
    "Approximate algorithms: Trade-off accuracy for speed",
    "Chain detection: Find chains of repeated similar patterns",
    "Ostinato: Find representative patterns across multiple time series"
]

for feature in advanced_features:
    print(f"  • {feature}")

# Practical applications
print(f"\nPractical applications:")
applications = [
    "IoT sensor data analysis and anomaly detection",
    "Financial market pattern recognition and trading signals",
    "Healthcare monitoring for vital sign pattern changes",
    "Manufacturing quality control and process monitoring", 
    "Speech and audio pattern recognition",
    "Environmental monitoring for climate pattern analysis",
    "Network traffic analysis for security monitoring",
    "Sports performance analysis and technique recognition"
]

for app in applications:
    print(f"  • {app}")

print(f"\nSTUMPY provides comprehensive tools for time series pattern analysis")

## Example 5: stumpy functionality test 5

In [ ]:
import stumpy
import numpy as np

# Example 5: Production-ready time series analysis workflows
print("=== Example 5: Production-Ready Time Series Analysis Workflows ===")

# Create realistic IoT sensor data scenario
np.random.seed(999)

class IoTSensorSimulator:
    """Simulate realistic IoT sensor data with patterns and anomalies"""
    
    def __init__(self, duration_hours=24, sample_rate_minutes=5):
        self.duration_hours = duration_hours
        self.sample_rate_minutes = sample_rate_minutes
        self.n_points = int((duration_hours * 60) / sample_rate_minutes)
        
    def generate_temperature_data(self):
        """Generate realistic temperature sensor data"""
        time_points = np.linspace(0, self.duration_hours, self.n_points)
        
        # Daily temperature cycle (24-hour pattern)
        daily_cycle = 20 + 8 * np.sin(2 * np.pi * time_points / 24 - np.pi/2)
        
        # Weather variations (slower changes)
        weather_noise = 3 * np.sin(2 * np.pi * time_points / 48) 
        
        # Random measurement noise
        measurement_noise = np.random.normal(0, 0.5, self.n_points)
        
        # Combine components
        temperature = daily_cycle + weather_noise + measurement_noise
        
        # Add some anomalies (equipment failures)
        anomaly_indices = np.random.choice(self.n_points, size=5, replace=False)
        for idx in anomaly_indices:
            # Create temperature spikes or drops
            anomaly_duration = min(12, self.n_points - idx)  # Up to 1 hour anomaly
            anomaly_magnitude = np.random.choice([-15, 15])  # Hot or cold anomaly
            temperature[idx:idx+anomaly_duration] += anomaly_magnitude
            
        return temperature, time_points, anomaly_indices

# Generate realistic sensor data
sensor_sim = IoTSensorSimulator(duration_hours=48, sample_rate_minutes=10)  # 2 days, 10-min sampling
temperature_data, time_points, true_anomalies = sensor_sim.generate_temperature_data()

print(f"IoT Temperature Sensor Data Simulation:")
print(f"  Duration: {sensor_sim.duration_hours} hours")
print(f"  Sample rate: {sensor_sim.sample_rate_minutes} minutes")
print(f"  Total data points: {len(temperature_data)}")
print(f"  Temperature range: {temperature_data.min():.1f}°C to {temperature_data.max():.1f}°C")
print(f"  Injected anomalies at positions: {sorted(true_anomalies)}")

# Production anomaly detection workflow
print(f"\nProduction Anomaly Detection Workflow:")
try:
    # Step 1: Choose appropriate window size (1-2 hours of data)
    window_hours = 1.0
    m = int(window_hours * 60 / sensor_sim.sample_rate_minutes)
    print(f"  Window size: {m} samples ({window_hours} hours)")
    
    # Step 2: Compute matrix profile for anomaly detection
    matrix_profile = stumpy.stump(temperature_data, m=m)
    distances = matrix_profile.P_  # float64; np.std on stump's object-dtype column raises TypeError (see Example 1)
    
    # Step 3: Statistical anomaly detection
    anomaly_threshold = np.mean(distances) + 2.5 * np.std(distances)
    detected_anomalies = np.where(distances > anomaly_threshold)[0]
    
    print(f"  Anomaly threshold: {anomaly_threshold:.4f}")
    print(f"  Detected anomalies: {len(detected_anomalies)} positions")
    print(f"  Anomaly positions: {detected_anomalies[:10]}")  # Show first 10
    
    # Step 4: Evaluate detection performance
    detection_accuracy = []
    for true_anomaly in true_anomalies:
        # Check if any detection is within reasonable distance
        close_detections = detected_anomalies[np.abs(detected_anomalies - true_anomaly) <= 6]  # Within 1 hour
        if len(close_detections) > 0:
            detection_accuracy.append(True)
        else:
            detection_accuracy.append(False)
            
    accuracy_rate = np.mean(detection_accuracy)
    print(f"  Detection accuracy: {accuracy_rate:.2%}")
    
except Exception as e:
    print(f"Error in anomaly detection workflow: {e}")
    raise

# Production pattern discovery for maintenance scheduling
print(f"\nPattern Discovery for Predictive Maintenance:")
try:
    # Find recurring patterns that might indicate maintenance needs
    m_pattern = 24  # Look for patterns over 4 hours
    
    # Use SCRUMP for faster approximate computation
    scrump_result = stumpy.scrump(temperature_data, m=m_pattern, percentage=0.2)
    scrump_distances = scrump_result.P_
    
    # Find top motifs (recurring patterns)
    motifs = stumpy.motifs(temperature_data, scrump_distances, max_motifs=3)
    
    print(f"  Pattern discovery window: {m_pattern} samples")
    print(f"  Top recurring patterns found: {len(motifs)}")
    
    for i, motif_group in enumerate(motifs):
        if len(motif_group) >= 2:
            pos1, pos2 = motif_group[0], motif_group[1]
            # Convert positions to time
            time1 = pos1 * sensor_sim.sample_rate_minutes / 60  # Hours
            time2 = pos2 * sensor_sim.sample_rate_minutes / 60  # Hours
            print(f"    Motif {i+1}: Similar patterns at {time1:.1f}h and {time2:.1f}h")

except Exception as e:
    print(f"Error in pattern discovery: {e}")
    raise

# Production monitoring dashboard simulation
print(f"\nReal-time Monitoring Dashboard Simulation:")
try:
    # Simulate streaming data processing
    initial_data = temperature_data[:200]  # Start with partial data
    streaming_mp = stumpy.stumpi(initial_data, m=20)
    
    print(f"  Initial monitoring window: {len(initial_data)} samples")
    
    # Simulate new data arriving in batches
    batch_size = 10
    batches_processed = 0
    alerts_triggered = 0
    
    for i in range(200, len(temperature_data), batch_size):
        batch_end = min(i + batch_size, len(temperature_data))
        new_batch = temperature_data[i:batch_end]
        
        # Process each new data point
        for new_point in new_batch:
            # Update streaming matrix profile
            streaming_mp.update(new_point)
            
            # Check for anomalies in real-time
            current_distance = streaming_mp.P_[-1]  # Latest distance
            if current_distance > anomaly_threshold:
                alerts_triggered += 1
        
        batches_processed += 1
        
        # Show status every few batches
        if batches_processed % 10 == 0:
            current_time = (i * sensor_sim.sample_rate_minutes) / 60
            print(f"    Processed {batches_processed} batches, time: {current_time:.1f}h, alerts: {alerts_triggered}")
    
    print(f"  Total batches processed: {batches_processed}")
    print(f"  Total alerts triggered: {alerts_triggered}")
    
except Exception as e:
    print(f"Error in streaming monitoring: {e}")
    raise

# Production deployment considerations
print(f"\nProduction Deployment Best Practices:")
best_practices = [
    "Data preprocessing: Handle missing values, outliers, and scaling",
    "Window size selection: Choose based on domain knowledge and patterns",
    "Algorithm selection: STUMP for accuracy, SCRUMP for speed",
    "Threshold tuning: Use historical data for anomaly threshold calibration",
    "Streaming processing: Use STUMPI for real-time applications",
    "Performance monitoring: Track computation time and memory usage",
    "Alert management: Implement alert filtering and escalation policies",
    "Model validation: Regular validation against ground truth data",
    "Data storage: Efficient storage of matrix profiles and metadata",
    "Scalability: Consider GPU acceleration for large datasets"
]

for practice in best_practices:
    print(f"  • {practice}")

# Performance and scalability guidelines
print(f"\nPerformance Guidelines:")
performance_tips = [
    "Time complexity: O(n²) for STUMP, O(n log n) for SCRUMP",
    "Memory usage: Store only essential matrix profile data",
    "Batch processing: Process data in chunks for memory efficiency", 
    "Parallel processing: Use distributed computing for multiple sensors",
    "Window size impact: Smaller windows = faster computation",
    "Approximate algorithms: Use SCRUMP with appropriate percentage",
    "GPU acceleration: Use GPU variants for datasets > 10K points",
    "Incremental updates: Use STUMPI to avoid recomputation"
]

for tip in performance_tips:
    print(f"  • {tip}")

print(f"\nSTUMPY enables robust, production-ready time series analysis systems")
print(f"Suitable for IoT monitoring, industrial automation, and predictive maintenance")